# HW3 — Two retrievers and a model: what RAG actually sends

**Name: Zhaiylgan Gulnaz** 
**Student ID: S23067637**
**Group: 10**
**Repository: https://github.com/KALDYBAKYZY/hw3-kaldybakyzy**

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

> Helps to write the code, because I do not have sufficient knowledge working with Python and to understand the tasks definition.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** (9 + 10 + 6) / 3 = 25 / 3 = **8.3333**

**Query terms:** how, does, rag, retrieve, information (5 terms)

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = 1.9459 |
| does | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = 1.9459 |
| rag | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = 1.9459 |
| information | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = 1.9459 |

**BM25 of each document** (write out the sum for every document, one line per term that contributes):

> Length normalisation k1·(1 − b + b·|d|/avgdl):
> - D1: 1.5 · (0.25 + 0.75 · 9 / 8.3333) = 1.5900
> - D2: 1.5 · (0.25 + 0.75 · 10 / 8.3333) = 1.7250
> - D3: 1.5 · (0.25 + 0.75 · 6 / 8.3333) = 1.1850
>
> No query term occurs in any document (f(t, D) = 0 for every term and every document), so every term of the sum is idf(t) · 0 · 2.5 / (0 + …) = 0. No term contributes:
> - score(D1) = 0
> - score(D2) = 0
> - score(D3) = 0
>
> Checked again: "retrieve" ≠ "retrieves" / "retrieval" as exact tokens, and "RAG" never appears — D2 spells it out as "Retrieval-Augmented Generation". The zeros are correct.

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.0000 | 1 (tie) |
| D2 | 0.0000 | 1 (tie) |
| D3 | 0.0000 | 1 (tie) |

## A1 (b) — BM25 with the stems from README §3

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> **(a):** no document is retrieved first. All three score 0, because BM25 only counts exact token matches and none of the five query words appears in any document: "retrieve" is not the same token as "retrieves" or "retrieval", and "RAG" is never written as an abbreviation — D2 spells it out as "Retrieval-Augmented Generation". The ranking is a three-way tie, so BM25 cannot tell the relevant document from the irrelevant ones.
>
> **(b):** D2 is first (0.6857). Stemming maps "retrieve", "retrieves" and "retrieval" to the same stem "retriev", so the query term now matches D2 twice. It is still the only term that matches: "rag", "how", "doe" and "inform" match nothing, so D2 wins on one word. D1 and D3 stay tied at 0.


‖Query‖ = √(0.8² + 0.2² + 0.5²) = √(0.64 + 0.04 + 0.25) = √0.93 = 0.9644
‖D1‖ = √(0.49 + 0.01 + 0.16) = √0.66 = 0.8124
‖D2‖ = √(0.81 + 0.09 + 0.36) = √1.26 = 1.1225
‖D3‖ = √(0.01 + 0.64 + 0.04) = √0.69 = 0.8307

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = 0.78 | 0.9644 | 0.8124 | 0.78 / (0.9644 · 0.8124) = 0.78 / 0.7835 = **0.9956** |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = 1.08 | 0.9644 | 1.1225 | 1.08 / (0.9644 · 1.1225) = 1.08 / 1.0825 = **0.9977** |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = 0.34 | 0.9644 | 0.8307 | 0.34 / (0.9644 · 0.8307) = 0.34 / 0.8011 = **0.4244** |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> D2 (cosine 0.9977). D1 is almost as close (0.9956, a difference of only 0.002), while D3 is far away (0.4244).

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> **vs A1 (a):** no. BM25 (a) gave all three documents 0, a three-way tie, so it had no ranking at all; cosine gives a clear order D2 > D1 > D3.
>
> **vs A1 (b):** the top document is the same (D2), but the rest differs. BM25 (b) ties D1 and D3 at 0, while cosine puts D1 almost level with D2 and D3 clearly last.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 only counts exact tokens, so it sees no link between "RAG" in the query and "Retrieval-Augmented Generation" in D2, or between "retrieve" and "retrieves"/"retrieval" until we stem by hand. "information" and "how does" match nothing anywhere. An embedding model compares meaning, so "RAG" and "Retrieval-Augmented Generation" land close together, and "retrieve information" is close to "retrieves relevant documents" without any shared token. The same applies to D1: it shares no word with the query, so BM25 scores it 0, yet "language models … answer questions" is about the same topic as RAG, so its vector is close to the query's. D3 ("embeddings represent text as numerical vectors") is a different topic, so it is far in both systems.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [35]:
import math, re

docs = {
    "D1": "Large language models can generate text and answer questions.",
    "D2": "Retrieval-Augmented Generation retrieves relevant documents before generating an answer.",
    "D3": "Embeddings represent text as numerical vectors.",
}
query = "How does RAG retrieve information?"
k1, b = 1.5, 0.75

def tokenize(text):
    text = text.lower()                       
    text = re.sub(r"[^\w\s-]", "", text)       
    return [t for t in re.split(r"[\s-]+", text) if t] 

STEMS = {"retrieve": "retriev", "retrieves": "retriev", "retrieval": "retriev",
         "generate": "gener", "generating": "gener", "generation": "gener",
         "information": "inform", "does": "doe"}

def stem(tokens):
    return [STEMS.get(t, t) for t in tokens]  

In [ ]:
def bm25_table(doc_tokens, query_tokens, title):
    print("=" * 70); print(title); print("=" * 70)
    N = len(doc_tokens)
    lengths = {d: len(t) for d, t in doc_tokens.items()}
    avgdl = sum(lengths.values()) / N

    print("\nТокены и длины документов:")
    for d, t in doc_tokens.items():
        print(f"  {d} |d|={lengths[d]}: {t}")
    print(f"\navgdl = ({' + '.join(str(l) for l in lengths.values())}) / {N} = {avgdl:.4f}")
    print(f"\nТермины запроса: {query_tokens}")

    print(f"\n{'term':<12}" + "".join(f"f(t,{d})".rjust(9) for d in doc_tokens) + "   n(t)     idf(t)")
    idf = {}
    for t in query_tokens:
        f = {d: toks.count(t) for d, toks in doc_tokens.items()}
        n = sum(1 for v in f.values() if v > 0)
        idf[t] = math.log((N - n + 0.5) / (n + 0.5))
        print(f"{t:<12}" + "".join(str(f[d]).rjust(9) for d in doc_tokens) + f"{n:>7}  {idf[t]:>9.4f}")

    print("\nBM25 по документам (вклад дают только термины с f>0):")
    scores = {}
    for d, toks in doc_tokens.items():
        norm = k1 * (1 - b + b * lengths[d] / avgdl)
        total = 0.0
        print(f"  {d}: k1*(1-b+b*|d|/avgdl) = 1.5*(0.25+0.75*{lengths[d]}/{avgdl:.4f}) = {norm:.4f}")
        for t in query_tokens:
            f = toks.count(t)
            if f == 0:
                continue
            part = idf[t] * f * (k1 + 1) / (f + norm)
            total += part
            print(f"     '{t}': {idf[t]:.4f} * {f}*2.5 / ({f} + {norm:.4f}) = {part:.4f}")
        scores[d] = total
        print(f"     score({d}) = {total:.4f}")

    print("\nРанжирование:")
    for r, (d, s) in enumerate(sorted(scores.items(), key=lambda x: -x[1]), 1):
        print(f"  {r}. {d}  {s:.4f}")
    return scores

In [17]:
doc_tok = {d: tokenize(t) for d, t in docs.items()}
q_tok = tokenize(query)

scores_a = bm25_table(doc_tok, q_tok, "A1 (a) — BM25 без стемминга")
scores_b = bm25_table({d: stem(t) for d, t in doc_tok.items()}, stem(q_tok),
                      "A1 (b) — BM25 со стеммингом")

A1 (a) — BM25 без стемминга

Токены и длины документов:
  D1 |d|=9: ['large', 'language', 'models', 'can', 'generate', 'text', 'and', 'answer', 'questions']
  D2 |d|=10: ['retrieval', 'augmented', 'generation', 'retrieves', 'relevant', 'documents', 'before', 'generating', 'an', 'answer']
  D3 |d|=6: ['embeddings', 'represent', 'text', 'as', 'numerical', 'vectors']

avgdl = (9 + 10 + 6) / 3 = 8.3333

Термины запроса: ['how', 'does', 'rag', 'retrieve', 'information']

term          f(t,D1)  f(t,D2)  f(t,D3)   n(t)     idf(t)
how                 0        0        0      0     1.9459
does                0        0        0      0     1.9459
rag                 0        0        0      0     1.9459
retrieve            0        0        0      0     1.9459
information         0        0        0      0     1.9459

BM25 по документам (вклад дают только термины с f>0):
  D1: k1*(1-b+b*|d|/avgdl) = 1.5*(0.25+0.75*9/8.3333) = 1.5900
     score(D1) = 0.0000
  D2: k1*(1-b+b*|d|/avgdl) = 1.5*(0.25+

In [41]:
q = [0.8, 0.2, 0.5]
vecs = {"D1": [0.7, 0.1, 0.4], "D2": [0.9, 0.3, 0.6], "D3": [0.1, 0.8, 0.2]}
norm = lambda v: math.sqrt(sum(x * x for x in v))

nq = norm(q)
print(f"‖Query‖ = sqrt({' + '.join(f'{x}²' for x in q)}) = {nq:.4f}")
cos = {}
for d, v in vecs.items():
    dot = sum(a * c for a, c in zip(q, v))          
    nd = norm(v)                                  
    cos[d] = dot / (nq * nd)                 
    print(f"{d}: dot = {' + '.join(f'{a}*{c}' for a, c in zip(q, v))} = {dot:.4f};"
          f"‖{d}‖ = {nd:.4f};  cos = {cos[d]:.4f}")

print("\nРанжирование:")
for r, (d, s) in enumerate(sorted(cos.items(), key=lambda x: -x[1]), 1):
    print(f"  {r}. {d}  {s:.4f}")

‖Query‖ = sqrt(0.8² + 0.2² + 0.5²) = 0.9644
D1: dot = 0.8*0.7 + 0.2*0.1 + 0.5*0.4 = 0.7800;‖D1‖ = 0.8124;  cos = 0.9956
D2: dot = 0.8*0.9 + 0.2*0.3 + 0.5*0.6 = 1.0800;‖D2‖ = 1.1225;  cos = 0.9977
D3: dot = 0.8*0.1 + 0.2*0.8 + 0.5*0.2 = 0.3400;‖D3‖ = 0.8307;  cos = 0.4244

Ранжирование:
  1. D2  0.9977
  2. D1  0.9956
  3. D3  0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [19]:
import json, re, os
import numpy as np

with open("data/corpus.jsonl", encoding="utf-8") as f:
    corpus = [json.loads(line) for line in f]

with open("data/queries.json", encoding="utf-8") as f:
    queries = json.load(f)["queries"]

ids    = [c["id"] for c in corpus]
texts  = [c["text"] for c in corpus]
titles = [c["title"] for c in corpus]

print("Чанков в корпусе:", len(corpus))
print("Запросов:", len(queries))
print("Пример чанка:", corpus[0]["id"], "|", corpus[0]["title"], "|", corpus[0]["text"][:80])

Чанков в корпусе: 74
Запросов: 8
Пример чанка: KZ-01 | Kazakhstan (part 1) | Kazakhstan, officially the Republic of Kazakhstan, is a landlocked country situa


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

>Lowercase, then re.findall(r"\w+"). No stemming, no stop-word removal. \w+ splits “GR-210” into “gr” and “210”, and it keeps Cyrillic/Kazakh words.

In [34]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\w+", text.lower())

bm25 = BM25Okapi([tokenize(t) for t in texts], k1=1.5, b=0.75)

def search_bm25(query, k=3):
    scores = bm25.get_scores(tokenize(query))      # score для каждого из 74 чанков
    top = np.argsort(-scores)[:k]                  # индексы k лучших
    return [(ids[i], float(scores[i])) for i in top]

print(search_bm25("What is form GR-305 used for?"))   # быстрая проверка

[('GO-08', 15.825491699499644), ('GO-12', 8.195256931904641), ('GO-06', 8.07003113015204)]


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [24]:
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()  

model = SentenceTransformer("google/embeddinggemma-300m", token=os.environ["HF_TOKEN"])

doc_emb = model.encode_document(texts)
print("Матрица эмбеддингов:", doc_emb.shape) 

def search_emb(query, k=3):
    q_emb = model.encode_query(query)                      
    scores = model.similarity(q_emb, doc_emb)[0].cpu().numpy() 
    top = np.argsort(-scores)[:k]
    return [(ids[i], float(scores[i])) for i in top]

print(search_emb("What is form GR-305 used for?"))

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 5506.91it/s]
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


Матрица эмбеддингов: (74, 768)
[('GO-08', 0.6955164670944214), ('GO-12', 0.46087244153022766), ('GO-23', 0.4504861831665039)]


## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | What happens to a GR-210 with a missing signature? | GO-07 | **BM25.** The query contains the exact form code "GR-210" and the word "signature", which appear in GO-07. The embedding model may blur GR-210 with the similar forms GR-104 and GR-305. |
| Q-10 | Can I take a break from university for a while and still receive the money afterwards? | GO-18 | **Embeddings.** GO-18 talks about "academic leave" and "payments resume", but the query uses none of those words ("break", "money", "afterwards"), so BM25 has nothing to match. The meaning is the same. |

In [25]:
my_queries = [
    {"id": "Q-09", "text": "What happens to a GR-210 with a missing signature?", "relevant": ["GO-07"]},
    {"id": "Q-10", "text": "Can I take a break from university for a while and still receive the money afterwards?", "relevant": ["GO-18"]},
]
all_queries = queries + my_queries
print(len(all_queries), "queries")

10 queries


## Results for all ten or more queries: top 3 from each retriever

In [26]:
def first_line(i, n=70):
    return f"{titles[i]} — {texts[i][:n]}..."

results = {} 
for q in all_queries:
    print("=" * 100)
    print(f"{q['id']}: {q['text']}")
    print(f"relevant: {q['relevant']}")
    results[q["id"]] = {}
    for name, search in [("BM25", search_bm25), ("EmbeddingGemma", search_emb)]:
        top = search(q["text"])
        hit = any(doc_id in q["relevant"] for doc_id, _ in top)
        results[q["id"]][name] = {"top": top, "hit": hit}
        print(f"\n  {name}   hit@3 = {'✓' if hit else '✗'}")
        for rank, (doc_id, score) in enumerate(top, 1):
            print(f"    {rank}. {doc_id:6} {score:7.3f}  {first_line(ids.index(doc_id))}")

print("\n" + "=" * 100)
for name in ["BM25", "EmbeddingGemma"]:
    total = sum(results[q["id"]][name]["hit"] for q in all_queries)
    print(f"{name}: hit@3 = {total}/{len(all_queries)}")

Q-01: What is form GR-305 used for?
relevant: ['GO-08', 'GO-12', 'GO-23']

  BM25   hit@3 = ✓
    1. GO-08   15.825  Form GR-305: appeals — Form GR-305 is used to appeal a decision of the Grant Office. It is th...
    2. GO-12    8.195  How to appeal a decision — A student who disagrees with a decision may appeal within 14 calendar ...
    3. GO-06    8.070  Form GR-104: the application form — Form GR-104 is the application form for the need-based study grant. It...

  EmbeddingGemma   hit@3 = ✓
    1. GO-08    0.696  Form GR-305: appeals — Form GR-305 is used to appeal a decision of the Grant Office. It is th...
    2. GO-12    0.461  How to appeal a decision — A student who disagrees with a decision may appeal within 14 calendar ...
    3. GO-23    0.450  Апелляция: кратко — Если студент не согласен с решением, он может подать апелляцию по форм...
Q-02: My family is poor. Is there financial help for students like me?
relevant: ['GO-01', 'GO-02', 'GO-03']

  BM25   hit@3 = ✗
    1. KZ

## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✓ | GO-08, GO-12, GO-23 | ✓ |
| Q-02 | KZ-17, GO-20, KZ-25 | ✗ | GO-01, GO-10, GO-20 | ✓ |
| Q-03 | GO-14, GO-02, GO-01 | ✓ | GO-14, GO-09, GO-11 | ✓ |
| Q-04 | GO-05, GO-17, GO-19 | ✗ | GO-02, GO-05, GO-17 | ✓ |
| Q-05 | KZ-05, KZ-06, KZ-11 | ✓ | KZ-05, KZ-06, KZ-03 | ✓ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✓ | KZ-37, KZ-38, KZ-10 | ✓ |
| Q-07 | GO-21, GO-16, GO-01 | ✗ | GO-15, GO-18, GO-01 | ✓ |
| Q-08 | GO-17, GO-05, GO-04 | ✓ | GO-17, GO-18, GO-15 | ✓ |
| Q-09 | GO-07, GO-13, GO-10 | ✓ | GO-07, GO-10, GO-13 | ✓ |
| Q-10 | GO-21, KZ-41, GO-16 | ✗ | GO-18, GO-17, GO-15 | ✓ |
| **hit@3 total** | | **6**/10 | | **10**/10 |

**Both retrievers agree:** which query, and why was it easy?

> **Q-06** ("Where did the Soviet Union test its nuclear weapons?"). Both put KZ-37 and KZ-38 (Semipalatinsk Test Site) at ranks 1 and 2 (BM25 19.09 / 17.17, embeddings 0.596 / 0.510). It was easy because the query and the chunk share the same rare words: "soviet", "union", "nuclear", "test" appear literally in KZ-38 ("The Soviet Union conducted 456 nuclear tests…"), so BM25 matches them, and the meaning is identical, so the embeddings agree too. Q-05 (Astana → KZ-05) is the same kind of easy case.

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

> In this run BM25 never beat the embeddings on hit@3 (6/10 vs 10/10), so there is no query where only BM25 succeeded. The closest case is **Q-09** ("What happens to a GR-210 with a missing signature?"), the query I predicted BM25 would win. BM25 matched the exact strings "gr", "210", "missing" and "signature" in GO-07 ("A GR-210 with a missing signature is returned unprocessed") and scored it 19.26, far ahead of GO-13 (13.22). My prediction was only half right: EmbeddingGemma also put GO-07 first (0.619). The exact form code is where BM25 is strongest, but here the code is also written in the chunk's title, so the embedding model did not confuse GR-210 with GR-104 or GR-305. Q-03 shows the same strength: BM25 matched the Kazakh tokens "дейін" and "керек" in GO-14 and gave every other chunk exactly 0, a much sharper separation than the embeddings (GO-14 0.399 vs irrelevant GO-11 0.301).

**Embeddings did better:** which query, and what could BM25 not match?

> **Q-07** ("Can the grant be sent to my mother's card?"). The answer is only in GO-15, which is in Russian ("перевод на счёт родителей или других лиц невозможен"). The query shares **no token at all** with GO-15, so BM25 scored it 0 and instead ranked GO-21 first (14.65), because GO-21 contains the English words "grant", "sent", "can", "be", which carry no meaning here. EmbeddingGemma is multilingual and put GO-15 first (0.422): "mother's card" and "счёт родителей" mean the same thing. The margin is small (GO-18 0.417, GO-01 0.411), though, so the embedding model was not confident.
>
> Q-02 and Q-10 are paraphrase failures for BM25: "poor… financial help" is never written in GO-01/GO-03 (they say "need-based", "income band"), and "take a break… receive the money" is never written in GO-18 ("academic leave", "payments resume"). BM25 matched only stop words, e.g. KZ-17 (Lake Balkhash) ranked first for Q-02 on "for", "like", "there", "is".
>
> **Q-04** is the trap: BM25 ranked GO-05 first (12.78), the **archived 2025 rule**, because it contains the literal phrase "the minimum GPA for the grant was 3.0". The correct chunk GO-02 never uses the word "minimum" (it says "at least 2.67"), so BM25 missed it. Embeddings put GO-02 first (0.665), but GO-05 is a close second (0.621), so it will still be sent to the model in Part C.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [27]:
import tiktoken
from openai import OpenAI

client = OpenAI()         
MODEL = "gpt-5.6-luna"
enc = tiktoken.get_encoding("o200k_base")
n_tok = lambda s: len(enc.encode(s))

with open("data/qa.json", encoding="utf-8") as f:
    qa = json.load(f)["questions"]

SYSTEM = (
    "Answer the question using only the provided context.\n"
    "If the answer cannot be found in the context, say that you do not know.\n"
    "Cite the context blocks you used, like [1] or [2][3]."
)

def context_blocks(chunk_ids):
    """Нумерованные блоки [1] [2] [3]: заголовок чанка + его текст."""
    blocks = []
    for n, cid in enumerate(chunk_ids, 1):
        i = ids.index(cid)
        blocks.append(f"[{n}] {titles[i]}\n{texts[i]}")
    return blocks

def build_messages(question, chunk_ids):
    context = "Context:\n\n" + "\n\n".join(context_blocks(chunk_ids))
    user = context + "\n\nQuestion: " + question
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}], context

def ask(messages):
    resp = client.chat.completions.create(model=MODEL, messages=messages)
    return resp.choices[0].message.content, resp.usage

## C1 — the complete prompt, printed before it is sent

In [28]:
q = qa[0]                                         
top = search_emb(q["question"])                   
chunk_ids = [cid for cid, _ in top]
messages, context = build_messages(q["question"], chunk_ids)

print("Retrieved:", top, "\n")
print("=" * 30, "FULL PROMPT, exactly as sent", "=" * 30)
for m in messages:
    print(f"--- {m['role'].upper()} ---")
    print(m["content"], "\n")

question_part = "\n\nQuestion: " + q["question"]
t_sys, t_q, t_ctx = n_tok(SYSTEM), n_tok(question_part), n_tok(context)
print("=" * 30, "TOKENS (tiktoken o200k_base)", "=" * 30)
print(f"System instructions : {t_sys}")
print(f"Question            : {t_q}")
print(f"Retrieved context   : {t_ctx}")
print(f"Total (sum)         : {t_sys + t_q + t_ctx}")
for n, block in enumerate(context_blocks(chunk_ids), 1):
    print(f"  chunk [{n}] {chunk_ids[n-1]}: {n_tok(block)} tokens, {len(block)} chars")
print(f"Context size: {t_ctx} tokens, {len(context)} characters, {len(chunk_ids)} chunks")

answer, usage = ask(messages)
print("\nusage.prompt_tokens  :", usage.prompt_tokens)
print("difference           :", usage.prompt_tokens - (t_sys + t_q + t_ctx))
print("\nANSWER:\n", answer)

Retrieved: [('GO-04', 0.5845215320587158), ('GO-03', 0.48417532444000244), ('GO-15', 0.4506281316280365)] 

============================== FULL PROMPT, exactly as sent ==============================
--- SYSTEM ---
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]. 

--- USER ---
Context:

[1] How much the grant pays
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] Income bands
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 

## C1 — the complete prompt, printed before it is sent

Question C-01, top 3 from EmbeddingGemma: GO-04 (0.585), GO-03 (0.484), GO-15 (0.451).

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 21 |
| Retrieved context | 304 |
| **Total input** | **369** |
| **`usage.prompt_tokens` from the response** | **378** |

- Number of retrieved chunks: 3
- Tokens per chunk: GO-04 = 99, GO-03 = 110, GO-15 = 93 (302 in the blocks; the remaining 2 are the "Context:" label and the separators)
- Total context size (tokens / characters): 304 tokens / 1,207 characters

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> The API counted 378 tokens, 9 more than my tiktoken sum of 369. tiktoken only counts the text I wrote. The API does not send that text as a plain string: it wraps each message in the chat format, with special tokens that mark where a message starts and ends and which role it has (system, user), and it adds tokens that prime the assistant's reply. That is about 3 tokens per message × 2 messages + 3 for the reply = 9, exactly the gap. The same overhead appeared in HW1.

**What exactly does retrieval add to the model's input?**

> Plain text, nothing else. Retrieval added 304 of the 378 input tokens (80%): the titles and texts of three chunks (GO-04, GO-03, GO-15), pasted into the user message as numbered blocks [1][2][3] before the question. The model gets no documents, no file access, no vectors and no scores. It sees one long prompt and continues it. The answer ("150,000 tenge … October and February [1]") is correct only because GO-04 was in that text.

## C2 — three systems, six questions

In [29]:
answers = {}
for q in qa:
    qid, question = q["id"], q["question"]
    bm_top  = search_bm25(question)
    emb_top = search_emb(question)
    answers[qid] = {
        "bm25_top": bm_top,
        "emb_top": emb_top,
        "LLM only": ask([{"role": "user", "content": question}])[0],
        "BM25 RAG": ask(build_messages(question, [c for c, _ in bm_top])[0])[0],
        "Embedding RAG": ask(build_messages(question, [c for c, _ in emb_top])[0])[0],
    }
    print("=" * 100)
    print(f"{qid}: {question}")
    print(f"REFERENCE: {q['reference']}")
    for system in ["LLM only", "BM25 RAG", "Embedding RAG"]:
        print(f"\n--- {system} ---\n{answers[qid][system]}")

C-01: How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.

--- LLM only ---
Which student funding scheme or country are you referring to? “Income band 2” varies by provider, and the amount and payment schedule depend on the academic year and whether the student lives at home or away.

--- BM25 RAG ---
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]

--- Embedding RAG ---
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]
C-02: My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305.

--- LLM only ---
The appeal d

## C2 — three systems, six questions

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ asks "which scheme or country?", no amount | ✓ 150,000 tenge, two instalments, October & February [1] | ✓ 150,000 tenge, 2 × 75,000, October & February [1] |
| C-02 | ✗ generic advice, "check your letter", no deadline or form | ✓ 14 calendar days, form GR-305 [2][3] | ✓ 14 calendar days, form GR-305 [2] |
| C-03 | ✗ "varies by grant… which school?" | ✓ 2.67 on a 4.0 scale [2] | ✓ 2.67 on a 4.0 scale [1] |
| C-04 | ✓ October 2017, first target 2025, now 2031 (from training data) | ✓ October 2017, 2025 → postponed to 2031 [1][2] | ✓ October 2017, now due 2031 [1][2] (omits 2025 and January 2021) |
| C-05 | ✓ 1963, "Alma-Ata Institute of National Economy" (Alma-Ata = old name of Almaty) | ✓ 1963, Almaty Institute of National Economy [1] | ✓ 1963, Almaty Institute of National Economy [1] |
| C-06 | ✗ invents a long answer about US Pell/TEACH grants and FAFSA | ✓ "I don't know the amount"; master's students have a separate scheme [3] | ✓ "I do not know"; separate scheme [1][3] |
| **agrees with `reference`** | **2**/6 | **6**/6 | **6**/6 |

## C3 — question 1 of 3: C-03

*"What is the minimum GPA needed to qualify for the need-based grant?"* Reference: at least 2.67 (the 3.0 in GO-05 is the archived 2025 rule).

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-16 (15.799), **GO-02 (14.059)**, GO-06 (13.852). GO-16 ("Combining the grant with other support") came first only because it repeats "need-based grant" several times; it says nothing about GPA. The relevant chunk GO-02 is second. The archived GO-05 is **not** in BM25's top 3 for this wording (unlike Q-04 in Part B, where "the minimum GPA for the grant" pulled GO-05 to rank 1).

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> **GO-02 (0.629)**, GO-20 (0.569), **GO-05 (0.545)**. The correct chunk is first, but the archived 2025 rule GO-05 is also in the top 3.

**3. What was sent to the model** (the context blocks):

> BM25 RAG: [1] GO-16 "Combining the grant with other support", [2] GO-02 "Who qualifies" ("their cumulative GPA is at least 2.67 on the 4.0 scale"), [3] GO-06 "Form GR-104: the application form".
>
> Embedding RAG: [1] GO-02 "Who qualifies" (2.67), [2] GO-20 "Who cannot apply", [3] GO-05 "Archived: the 2025 scheme (no longer in force)" ("Under the 2025 scheme the minimum GPA for the grant was 3.0 … These rules were replaced on 1 June 2026").

**4. How retrieval changed the answer:**

> Without retrieval the model could not answer at all ("The minimum GPA varies by the specific need-based grant and institution. Which grant or school are you referring to?"), because this handbook is fictional and not in its training data. With retrieval both RAG systems gave the correct, specific number: "2.67 on a 4.0 scale", citing the block that held GO-02 ([2] for BM25, [1] for embeddings). Retrieval made the answer correct, specific and grounded.

**5. What was missed, and was it a retrieval problem or a generation problem?**

> Nothing was missed in the final answers, but the embedding context was a trap. It contained two conflicting numbers: 2.67 in [1] and 3.0 in [3], where the 3.0 sits next to the very words of the question ("the minimum GPA for the grant was 3.0"). Retrieval was therefore only partly clean: it sent an outdated rule along with the right one. Generation handled it: the model used only [1] and ignored [3], because GO-05 states openly "ARCHIVED … do not apply to any application made for the 2026 academic year". BM25 sent irrelevant chunks (GO-16, GO-06) but not the conflicting one, so its context was noisier but safer. Both answers are correct, so there is no failure here. The risk is that if GO-05 had not been labelled "ARCHIVED", the model would have had no way to choose between 2.67 and 3.0.

In [30]:
def explain(qid):
    q = next(x for x in qa if x["id"] == qid)
    a = answers[qid]
    print(f"{qid}: {q['question']}\nREFERENCE: {q['reference']}   relevant: {q['relevant']}\n")
    print("1. BM25 top 3:          ", [(c, round(s, 3)) for c, s in a["bm25_top"]])
    print("2. EmbeddingGemma top 3:", [(c, round(s, 3)) for c, s in a["emb_top"]])
    for name, key in [("BM25", "bm25_top"), ("Embedding", "emb_top")]:
        print(f"\n3. Context sent to {name} RAG:")
        print("\n\n".join(context_blocks([c for c, _ in a[key]])))
    for system in ["LLM only", "BM25 RAG", "Embedding RAG"]:
        print(f"\n--- {system} ---\n{a[system]}")

explain("C-03")

C-03: What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)   relevant: ['GO-02']

1. BM25 top 3:           [('GO-16', 15.799), ('GO-02', 14.059), ('GO-06', 13.852)]
2. EmbeddingGemma top 3: [('GO-02', 0.629), ('GO-20', 0.569), ('GO-05', 0.545)]

3. Context sent to BM25 RAG:
[1] Combining the grant with other support
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] Who qualifies
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or i

## C3 — question 2 of 3: C-04

*"When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?"* Reference: a presidential decree of October 2017 ordered the change for 2025; in January 2021 the target was moved to 2031.

**1. What BM25 retrieved** (top 3, ids and scores):

> **KZ-43 (25.966)**, KZ-14 (24.062), KZ-06 (16.835). The relevant chunk is first. KZ-14 ("Kazakh language (part 3)") is not in `relevant` but covers the same decree. KZ-06 (Almaty) is noise, matched on "Kazakhstan" and "Cyrillic"-free words like "when", "now".

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> KZ-14 (0.720), **KZ-43 (0.693)**, KZ-05 (0.434). Both language chunks again, in the opposite order; KZ-05 (Astana) is noise.

**3. What was sent to the model** (the context blocks):

> BM25 RAG: [1] KZ-43 "Kazakh alphabets", [2] KZ-14 "Kazakh language (part 3)", [3] KZ-06 "Almaty (part 1)".
>
> Embedding RAG: [1] KZ-14 "Kazakh language (part 3)", [2] KZ-43 "Kazakh alphabets", [3] KZ-05 "Astana".
>
> Both contexts contain the full answer, in KZ-43: "In October 2017, a presidential decree in Kazakhstan ordered a transition from the Cyrillic to Latin script to be implemented by 2025. In January 2021, the target year for finishing the transition was pushed back to 2031."

**4. How retrieval changed the answer:**

> Very little, because this is a real Wikipedia fact the model already knew. LLM only, with no context, already answered "October 2017 … original target 2025 … now 2031". BM25 RAG gave the same content plus citations [1][2]: more grounded, not more accurate. Embedding RAG gave a **shorter** answer than the model with no context at all: "October 2017 … now due to finish in 2031 [1][2]". It dropped the original 2025 target and the January 2021 change.

**5. What was missed, and was it a retrieval problem or a generation problem?**

> Embedding RAG missed the original 2025 deadline and the date of the change (January 2021). This is a **generation problem**, not a retrieval problem: both facts were in its context, in block [2] (KZ-43): "to be implemented by 2025. In January 2021, the target year … was pushed back to 2031", and block [1] (KZ-14) also says "by 2025". The model had the facts and left them out. A likely reason is the order: embeddings put KZ-14 first, which says only "scheduled to be phased in from 2023 to 2031" and gives no date for the change, while BM25 put KZ-43 first and its answer included the postponement. The answer is not wrong, since it does answer both parts of the question, but it is less complete than the reference and than the no-context answer.

In [31]:
def explain(qid):
    q = next(x for x in qa if x["id"] == qid)
    a = answers[qid]
    print(f"{qid}: {q['question']}\nREFERENCE: {q['reference']}   relevant: {q['relevant']}\n")
    print("1. BM25 top 3:          ", [(c, round(s, 3)) for c, s in a["bm25_top"]])
    print("2. EmbeddingGemma top 3:", [(c, round(s, 3)) for c, s in a["emb_top"]])
    for name, key in [("BM25", "bm25_top"), ("Embedding", "emb_top")]:
        print(f"\n3. Context sent to {name} RAG:")
        print("\n\n".join(context_blocks([c for c, _ in a[key]])))
    for system in ["LLM only", "BM25 RAG", "Embedding RAG"]:
        print(f"\n--- {system} ---\n{a[system]}")

explain("C-04")

C-04: When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
REFERENCE: A presidential decree of October 2017 ordered the change for 2025; in January 2021 the target was moved to 2031.   relevant: ['KZ-43']

1. BM25 top 3:           [('KZ-43', 25.966), ('KZ-14', 24.062), ('KZ-06', 16.835)]
2. EmbeddingGemma top 3: [('KZ-14', 0.72), ('KZ-43', 0.693), ('KZ-05', 0.434)]

3. Context sent to BM25 RAG:
[1] Kazakh alphabets
The Kazakh language has been written mainly in four scripts at various points of time—Old Turkic, Cyrillic, Latin, and Arabic—each having a distinct alphabet. The Arabic script is used in Iran, Afghanistan, and China, while the Cyrillic script is used in Kazakhstan, Kyrgyzstan, Russia, and Mongolia. In October 2017, a presidential decree in Kazakhstan ordered a transition from the Cyrillic to Latin script to be implemented by 2025. In January 2021, the target year for finishing the transition was pushed ba

## C3 — question 3 of 3: C-06

*"I am a master's student. How much need-based grant can I get?"* Reference: the corpus does not say; GO-20 says master's students have a separate scheme that the handbook does not describe. The only correct answer is "I don't know".

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-16 (16.996), GO-01 (14.173), **GO-20 (13.720)**. GO-16 and GO-01 came first because they repeat "need-based grant"; the word "master's" appears only in GO-20, which still made the top 3.

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> **GO-20 (0.666)**, GO-16 (0.616), GO-01 (0.554). The same three chunks, but the relevant one is first: the embedding model matched the meaning "a master's student asking if they are eligible" to "Who cannot apply".

**3. What was sent to the model** (the context blocks):

> BM25 RAG: [1] GO-16 "Combining the grant with other support", [2] GO-01 "About the need-based study grant 2026", [3] GO-20 "Who cannot apply".
>
> Embedding RAG: [1] GO-20 "Who cannot apply", [2] GO-16 "Combining the grant with other support", [3] GO-01 "About the need-based study grant 2026".
>
> The key sentence, in both contexts: "Master's and doctoral students are supported by a separate scheme, which this handbook does not describe; they should ask their own department for its rules."

**4. How retrieval changed the answer:**

> This is where retrieval mattered most. Without context the model **hallucinated**: it assumed the user was in the U.S. and gave a long answer about Pell Grants, TEACH Grants, FAFSA and "$20,500 per academic year", none of which has anything to do with this grant office. With context both RAG systems answered correctly: "I don't know the amount … master's students are covered by a separate scheme and should ask their department for its rules", citing GO-20 ([3] for BM25, [1][3] for embeddings). Retrieval turned a confident invented answer into an honest, grounded "I don't know".

**5. What was missed, and was it a retrieval problem or a generation problem?**

> Nothing was missed: both RAG answers agree with the reference. Retrieval worked (GO-20 was in both contexts), and generation followed the system instruction "If the answer cannot be found in the context, say that you do not know." The context also held a tempting wrong number: GO-16 says support "is capped at 400,000 tenge", and a weaker model could have answered "up to 400,000 tenge". Ours did not use it. The only failure on this question is the **LLM-only** one: with no retrieval at all, the model had nothing to say "I don't know" about, so it invented an answer.

In [32]:
def explain(qid):
    q = next(x for x in qa if x["id"] == qid)
    a = answers[qid]
    print(f"{qid}: {q['question']}\nREFERENCE: {q['reference']}   relevant: {q['relevant']}\n")
    print("1. BM25 top 3:          ", [(c, round(s, 3)) for c, s in a["bm25_top"]])
    print("2. EmbeddingGemma top 3:", [(c, round(s, 3)) for c, s in a["emb_top"]])
    for name, key in [("BM25", "bm25_top"), ("Embedding", "emb_top")]:
        print(f"\n3. Context sent to {name} RAG:")
        print("\n\n".join(context_blocks([c for c, _ in a[key]])))
    for system in ["LLM only", "BM25 RAG", "Embedding RAG"]:
        print(f"\n--- {system} ---\n{a[system]}")

explain("C-06")

C-06: I am a master's student. How much need-based grant can I get?
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.   relevant: ['GO-20']

1. BM25 top 3:           [('GO-16', 16.996), ('GO-01', 14.173), ('GO-20', 13.72)]
2. EmbeddingGemma top 3: [('GO-20', 0.666), ('GO-16', 0.616), ('GO-01', 0.554)]

3. Context sent to BM25 RAG:
[1] Combining the grant with other support
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] About the need-based study grant 2026
The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for 

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 is lexical: it scores a chunk by counting exact tokens it shares with the query, weighted by how rare each token is (idf) and normalised by length. Embedding retrieval is semantic: the query and every chunk are turned into vectors (768 numbers for EmbeddingGemma), and the score is how close the vectors are, so chunks can match by meaning without sharing a single word. In Part A this was the whole story: BM25 scored all three documents 0 because "RAG" ≠ "Retrieval-Augmented Generation", while the vectors still ranked D2 first.

**2. When can BM25 perform better than embeddings?**

> When the query contains an exact, rare string that must match literally: a form code, a number, a name, a rare term. In Q-09, BM25 matched "gr", "210", "missing", "signature" and scored GO-07 at 19.26 against 13.22 for the next chunk; in Q-03 it gave GO-14 10.70 and every other chunk exactly 0. It needs no model, runs instantly and is easy to explain. In my run it never beat embeddings on hit@3, but it gave the sharpest separation exactly on these exact-string queries.

**3. When can embeddings perform better than BM25?**

> When the query and the answer use different words for the same thing: paraphrases, synonyms and other languages. Q-07 ("my mother's card") shares no token with the Russian GO-15 ("перевод на счёт родителей… невозможен"): BM25 scored it 0, while embeddings ranked it first (0.422). Q-02 ("poor… financial help" vs "need-based", "income band") and Q-10 ("take a break… the money" vs "academic leave… payments") are the same kind of case: BM25 missed all three, embeddings hit all three.

**4. Why is cosine similarity used with embeddings?**

> An embedding model puts the meaning of a text in the direction of its vector, not in its length, and length can vary with things like text size. Cosine measures only the angle between two vectors (dot product divided by both lengths), so it compares direction and ignores magnitude, giving a score from −1 to 1 that is comparable across chunks of different lengths. In A2, D2 had the largest dot product (1.08) partly because its vector is longest (1.1225); cosine removes that effect, which is why D1 (0.9956) came out almost level with D2 (0.9977).

**5. What information is actually sent to an LLM in a RAG system?**

> Only text: a system message, the retrieved chunks pasted in as numbered blocks, and the question. In my C1 example this was 44 tokens of instructions, 304 tokens of context (GO-04, GO-03, GO-15: 99 + 110 + 93 tokens, 1,207 characters) and 21 tokens of question, 378 prompt tokens in total by the API's count. No scores, vectors, ids or files are sent unless the code writes them into the prompt.

**6. Does the LLM search the original documents itself?**

> No. My code does the search (BM25 or EmbeddingGemma, on my machine), picks the top 3 and pastes them into the prompt. The model never sees the other 71 chunks and cannot ask for them; it only continues the text it receives. That is why LLM only could not answer C-01, C-02 or C-03, and invented an answer about U.S. grants for C-06: without retrieved text it has only its training data.

**7. What happens if retrieval returns an irrelevant chunk?**

> At best it wastes tokens; at worst it misleads the model. Irrelevant noise was common in my runs (GO-16 ranked first for C-03 and C-06 with BM25; KZ-06 Almaty and KZ-05 Astana came back for C-04), and the model ignored it. The dangerous case is a chunk that looks relevant but is wrong: for C-03 the embedding context included the archived GO-05 ("the minimum GPA for the grant was 3.0") next to the correct 2.67. The model chose correctly only because GO-05 says "ARCHIVED". If the relevant chunk is missing entirely, the model can only say "I don't know" or, if it ignores the instruction, guess.



**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

> I ran C-03 ("What is the minimum GPA needed to qualify for the need-based grant?") with EmbeddingGemma:
>
> | K | Chunks sent | Context tokens | `usage.prompt_tokens` | Archived GO-05 in context? | Answer |
> |---|---|---:|---:|:---:|---|
> | 1 | GO-02 | 110 | 180 | no | 2.67 on a 4.0 scale [1] ✓ |
> | 3 | GO-02, GO-20, GO-05 | 276 | 346 | **yes** | 2.67 on a 4.0 scale [1] ✓ |
> | 8 | GO-02, GO-20, GO-05, GO-16, GO-01, GO-06, GO-19, GO-17 | 746 | 816 | **yes** | 2.67 on a 4.0 scale [1] ✓ |
>
> The answer did not change, but the input did: from K = 1 to K = 8 the prompt grew 4.5× (180 → 816 tokens) and only the first chunk was ever cited. From K = 3 on, the context also contained the conflicting archived rule (3.0), so the answer was correct only because the model noticed the "ARCHIVED" label. Here K = 1 was the cheapest and the cleanest.
>
> A larger K **helps** when the answer is spread over several chunks, or when the right chunk is not ranked first: for C-06, BM25 put GO-20 at rank 3, so K = 1 would have sent only GO-16 and the model could not have said why a master's student is excluded; C-01 needs both the amount (GO-04) and the payment months. A larger K **hurts** when it adds noise and look-alike chunks (GO-05 here) that the model can confuse with the answer, and it always costs more tokens, latency and money. So K is a trade-off between recall and precision.

In [33]:
q = next(x for x in qa if x["id"] == "C-03")
for k in [1, 3, 8]:
    top = search_emb(q["question"], k=k)
    chunk_ids = [c for c, _ in top]
    messages, context = build_messages(q["question"], chunk_ids)
    answer, usage = ask(messages)
    print("=" * 90)
    print(f"K = {k}: {chunk_ids}")
    print(f"context: {n_tok(context)} tokens | usage.prompt_tokens: {usage.prompt_tokens}")
    print("GO-02 (2.67) in context:", "GO-02" in chunk_ids, "| GO-05 (archived 3.0) in context:", "GO-05" in chunk_ids)
    print("ANSWER:", answer)

K = 1: ['GO-02']
context: 110 tokens | usage.prompt_tokens: 180
GO-02 (2.67) in context: True | GO-05 (archived 3.0) in context: False
ANSWER: The minimum GPA needed is **2.67 on a 4.0 scale**. [1]
K = 3: ['GO-02', 'GO-20', 'GO-05']
context: 276 tokens | usage.prompt_tokens: 346
GO-02 (2.67) in context: True | GO-05 (archived 3.0) in context: True
ANSWER: The minimum GPA needed is **2.67 on a 4.0 scale**. [1]
K = 8: ['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17']
context: 746 tokens | usage.prompt_tokens: 816
GO-02 (2.67) in context: True | GO-05 (archived 3.0) in context: True
ANSWER: The minimum GPA needed to qualify for the 2026 need-based grant is **2.67 on a 4.0 scale**. [1]


**9. Is a vector database required to build a RAG system?**

> No. My notebook used no vector database: the 74 chunk embeddings were kept in memory in one array, `doc_emb`, of shape (74, 768), and every query was compared with all 74 vectors by brute force with `model.similarity`. BM25 needed no vectors at all. A vector database becomes useful when there are millions of chunks, when you need fast approximate search, persistence or metadata filters, but for 74 chunks an array is enough.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?**

> EmbeddingGemma. On my ten queries its hit@3 was **10/10**, against **6/10** for BM25. BM25 failed on Q-02 and Q-10 (paraphrase), Q-07 (the answer is only in Russian) and Q-04 (it ranked the archived GO-05 first because of the literal phrase "minimum GPA for the grant"). In C2 both RAG systems agreed with the reference on **6/6** questions (LLM only: 2/6), because the relevant chunk usually made the top 3 for both and the model coped with noise. The embeddings also more often ranked the right chunk first (C-03: GO-02 first vs second for BM25; C-06: GO-20 first vs third). Their weakness is that they also pull in look-alike chunks, such as the archived GO-05 in C-03.

---
# Conclusion
> In this homework, I compared two retrieval methods: BM25 and embedding search. I saw that BM25 works well with exact words, names, and codes, while embedding search is better for meaning, paraphrases, and different languages. This was clear in the tests: on my ten queries BM25 had hit@3 of 6/10 and EmbeddingGemma 10/10. BM25 failed when the query used different words than the answer (Q-02, Q-10) or when the answer was only in Russian (Q-07). I also learned that the model alone cannot answer many questions well, because it does not know the real context: LLM only agreed with the reference on 2/6 questions, and both RAG systems on 6/6. With RAG, the model gets the most relevant text first, and this helps it give more correct and grounded answers. I also saw that too many chunks can add noise: with K = 8 the prompt grew from 180 to 816 tokens, and the archived GO-05 rule came into the context. Next, I would build a hybrid retriever that combines BM25 and embeddings, because in my results they were strong in different places: embeddings for paraphrases and other languages, BM25 for exact codes like GR-210. I would also mark outdated chunks like GO-05 as archived and filter them out before retrieval, so the model never has to choose between an old rule and a new one. Overall, this homework showed that good retrieval is very important for a strong RAG system, and that the best answer comes from combining a good retriever with a careful language model.